# One Model Against All Benchmarks

This notebook runs one configurable model against every benchmark registered in `runners.full_suite.FULL_BENCHMARK_CLASSES`. It is set up for RunPod and local execution, records failures per benchmark, and resumes without overwriting completed results.

The default model is Qwen2.5-VL 3B. Set `DATASET_SMOKE_TEST=1` to use the repository's existing open-weight LLaVA-Gemma 2B model for a lighter end-to-end test. Even with one sample per benchmark, this run downloads many datasets and includes image, multi-image, and video tasks. Set `HF_TOKEN` in the RunPod environment for gated datasets such as ImageNet.

LSUN, TAD66K, and ShareGPT-4o-Image now use lightweight Hugging Face access by default: LSUN caches small validation LMDB archives, TAD66K range-reads only requested JPEG members, and ShareGPT streams WebDataset rows. The corresponding `*_ROOT` variables remain optional local-data overrides.

## 1. Repository and run configuration

In [1]:
from pathlib import Path
import getpass
import json
import os
import subprocess
import sys

REPO_URL = "https://github.com/samhatescoding/transformers.git"
REPO_BRANCH = "main"

candidate_roots = [Path.cwd(), *Path.cwd().parents]
REPO_ROOT = next((path for path in candidate_roots if (path / "models").is_dir()), None)
if REPO_ROOT is None:
    clone_parent = Path("/workspace") if Path("/workspace").is_dir() else Path.cwd()
    REPO_ROOT = clone_parent / "transformers"
    if REPO_ROOT.exists() and not (REPO_ROOT / "models").is_dir():
        raise RuntimeError(f"Clone target exists but is not this repository: {REPO_ROOT}")
    if not REPO_ROOT.exists():
        subprocess.run(
            ["git", "clone", "--branch", REPO_BRANCH, "--single-branch", REPO_URL, str(REPO_ROOT)],
            check=True,
        )
REPO_ROOT = REPO_ROOT.resolve()

requirements_path = REPO_ROOT / "requirements.txt"
%pip install -q -r "$requirements_path"
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

import pandas as pd
import torch
from huggingface_hub import login

DATASET_SMOKE_TEST = os.getenv("DATASET_SMOKE_TEST", "0") == "1"

# Best practice on RunPod: add HF_TOKEN under Pod > Environment Variables.
# If it is absent, this prompt is hidden and only lasts for the current kernel.
HF_TOKEN = os.getenv("HF_TOKEN") or os.getenv("HUGGING_FACE_HUB_TOKEN")
if not HF_TOKEN and not DATASET_SMOKE_TEST:
    entered_token = getpass.getpass("HF token (press Enter to continue without one): ").strip()
    if entered_token:
        HF_TOKEN = entered_token
        os.environ["HF_TOKEN"] = entered_token
if HF_TOKEN:
    login(token=HF_TOKEN, add_to_git_credential=False)

NUM_SAMPLES = 1
OVERWRITE = False
RUN_NAME = "dataset_smoke_test" if DATASET_SMOKE_TEST else "qwen25vl3b_all_benchmarks"
OUTPUT_DIR = REPO_ROOT / "results" / RUN_NAME
SUMMARY_PATH = OUTPUT_DIR / "run_summary.json"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

print(f"Repository: {REPO_ROOT}")
print(f"Results: {OUTPUT_DIR}")
print(f"CUDA available: {torch.cuda.is_available()}")
print(f"GPU: {torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'none'}")
print(f"LLaVA-Gemma 2B smoke test: {DATASET_SMOKE_TEST}")
print(f"HF authentication: {'configured' if HF_TOKEN else 'not configured'}")
for variable in ("LSUN_ROOT", "TAD66K_ROOT", "SHAREGPT4O_IMAGE_ROOT"):
    print(f"{variable}: {'configured (local override)' if os.getenv(variable) else 'using Hugging Face source'}")

Repository: C:\Users\Samuel\Documents\GitHub\transformers
Results: C:\Users\Samuel\Documents\GitHub\transformers\results\dataset_smoke_test
CUDA available: False
GPU: none
LLaVA-Gemma 2B smoke test: True
HF authentication: not configured
LSUN_ROOT: using Hugging Face source
TAD66K_ROOT: using Hugging Face source
SHAREGPT4O_IMAGE_ROOT: using Hugging Face source


C:\Users\Samuel\AppData\Roaming\Python\Python314\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


## 2. Load the selected model

Replace `Qwen25VL3B` with another public model class if needed. For API-backed wrappers, set the corresponding provider key before executing this cell.

In [2]:
from models import LlavaGemma2B, Qwen25VL3B
from runners import ModelRun

if DATASET_SMOKE_TEST:
    MODEL_NAME = "llava-gemma-2b"
    model_run = ModelRun.from_factory(
        MODEL_NAME,
        LlavaGemma2B,
        max_new_tokens=32,
    )
else:
    MODEL_NAME = "qwen2.5-vl-3b-instruct"
    model_run = ModelRun.from_factory(
        MODEL_NAME,
        Qwen25VL3B,
        max_new_tokens=32,
    )
model_run

Loading LLaVA processor...


The image processor of type `CLIPImageProcessor` is now loaded as a fast processor by default, even if the model checkpoint was saved with a slow processor. This is a breaking change and may produce slightly different outputs. To continue using the slow processor, instantiate this class with `use_fast=False`. 


Loading LLaVA model...


Loading weights:   0%|          | 0/559 [00:00<?, ?it/s]

Loading weights:   0%|          | 1/559 [00:05<50:56,  5.48s/it]

...

Loading weights:  89%|████████▉ | 497/559 [00:08<00:00, 238.87it/s]

Loading weights:  97%|█████████▋| 545/559 [00:08<00:00, 245.77it/s]

Loading weights: 100%|██████████| 559/559 [00:08<00:00, 62.16it/s] 

Device map: None


ModelRun(name='llava-gemma-2b', model=<models.llava_gemma_2b.LlavaGemma2B object at 0x000001CEB5412510>, load_time_seconds=16.250638900033664)

## 3. Instantiate every registered benchmark

In [3]:
from runners.full_suite import FULL_BENCHMARK_CLASSES

print(f"Benchmarks: {len(FULL_BENCHMARK_CLASSES)}")
[benchmark_class.benchmark_name for benchmark_class in FULL_BENCHMARK_CLASSES]

Benchmarks: 40


['blip3o_60k',
 'conceptual_captions',
 'conceptual_captions_caption',
 'cityscapes',
 'dfdc',
 'diffusiondb',
 'docvqa',
 'fairface',
 'fashion_mnist',
 'flickr30k',
 'flickr30k_entities',
 'flyingthings3d',
 'gqa',
 'hdtf',
 'hq_edit',
 'imagenet1k',
 'imgedit',
 'inaturalist',
 'internvid',
 'kinetics',
 'laion400m',
 'laion5b',
 'lsun',
 'lvis',
 'mscoco',
 'mscoco_caption',
 'mvtec_ad',
 'magicbrush',
 'openimages_v4',
 'openimages_v4_detection',
 'openvid1m',
 'pick_a_pic',
 'places',
 'sharegpt4o_image',
 'tad66k',
 'textcaps',
 'ucf101',
 'visual_cot',
 'visual_genome',
 'vqav2']

## 4. Run the complete matrix

With one model, the number of pairs equals the number of registered benchmarks. Each pair is executed through the canonical `run_benchmark_matrix` runner. Calling it one pair at a time lets the notebook record unavailable datasets in `run_summary.json` and resume without rerunning completed results.

In [4]:
import traceback

from runners import BenchmarkRun, run_benchmark_matrix

summaries = []
for benchmark_class in FULL_BENCHMARK_CLASSES:
    benchmark_name = str(benchmark_class.benchmark_name)
    result_path = OUTPUT_DIR / f"{MODEL_NAME}_{benchmark_name}.json"

    if result_path.exists() and not OVERWRITE:
        entry = {
            "model": MODEL_NAME,
            "benchmark": benchmark_name,
            "status": "skipped",
            "results_path": str(result_path),
        }
    else:
        print(f"[RUN] {MODEL_NAME} / {benchmark_name}", flush=True)
        try:
            benchmark = benchmark_class(streaming=True)
            model_run.model.max_new_tokens = benchmark.default_max_new_tokens
            sample_count = min(NUM_SAMPLES, 9) if benchmark_name == "ucf101" else NUM_SAMPLES
            pair_summary = run_benchmark_matrix(
                models=[model_run],
                benchmark_runs=[BenchmarkRun(benchmark=benchmark, num_samples=sample_count)],
                output_dir=OUTPUT_DIR,
            )[0]
            entry = {**pair_summary, "status": "ok"}
        except Exception as exc:
            entry = {
                "model": MODEL_NAME,
                "benchmark": benchmark_name,
                "status": "error",
                "error": f"{type(exc).__name__}: {exc}",
                "traceback": traceback.format_exc(),
            }

    summaries.append(entry)
    SUMMARY_PATH.write_text(json.dumps(summaries, indent=2), encoding="utf-8")
    print(f"[{entry['status'].upper()}] {MODEL_NAME} / {benchmark_name}", flush=True)

summary_df = pd.DataFrame(summaries)
summary_df

[RUN] llava-gemma-2b / blip3o_60k


[ERROR] llava-gemma-2b / blip3o_60k


[RUN] llava-gemma-2b / conceptual_captions



[OK] llava-gemma-2b / conceptual_captions


[RUN] llava-gemma-2b / conceptual_captions_caption


...

[RUN] llava-gemma-2b / visual_genome


[ERROR] llava-gemma-2b / visual_genome


[RUN] llava-gemma-2b / vqav2



[OK] llava-gemma-2b / vqav2


,model,benchmark,status,error,traceback,num_samples,max_new_tokens,results_path
0,llava-gemma-2b,blip3o_60k,error,ValueError: Could not find two curated attribu...,"Traceback (most recent call last):\n File ""C:...",NaN,NaN,NaN
1,llava-gemma-2b,conceptual_captions,ok,NaN,NaN,1.0,16.0,C:\Users\Samuel\Documents\GitHub\transformers\...
2,llava-gemma-2b,conceptual_captions_caption,ok,NaN,NaN,1.0,24.0,C:\Users\Samuel\Documents\GitHub\transformers\...
3,llava-gemma-2b,cityscapes,ok,NaN,NaN,1.0,16.0,C:\Users\Samuel\Documents\GitHub\transformers\...
4,llava-gemma-2b,dfdc,ok,NaN,NaN,1.0,16.0,C:\Users\Samuel\Documents\GitHub\transformers\...
5,llava-gemma-2b,diffusiondb,error,RuntimeError: Dataset scripts are no longer su...,"Traceback (most recent call last):\n File ""C:...",NaN,NaN,NaN
6,llava-gemma-2b,docvqa,ok,NaN,NaN,1.0,16.0,C:\Users\Samuel\Documents\GitHub\transformers\...
7,llava-gemma-2b,fairface,ok,NaN,NaN,1.0,16.0,C:\Users\Samuel\Documents\GitHub\transformers\...
8,llava-gemma-2b,fashion_mnist,ok,NaN,NaN,1.0,16.0,C:\Users\Samuel\Documents\GitHub\transformers\...
9,llava-gemma-2b,flickr30k,ok,NaN,NaN,1.0,24.0,C:\Users\Samuel\Documents\GitHub\transformers\...


## 5. Inspect aggregate results

In [5]:
rows = []
for path in sorted(OUTPUT_DIR.glob("*.json")):
    if path == SUMMARY_PATH:
        continue
    payload = json.loads(path.read_text(encoding="utf-8"))
    report = payload["report"]
    stats = report.get("stats", {})
    rows.append(
        {
            "model": payload["model"],
            "benchmark": payload["benchmark"],
            "samples": report.get("num_samples"),
            "success_count": stats.get("success_count"),
            "failure_count": stats.get("failure_count"),
            "wall_clock_seconds": stats.get("wall_clock_time_seconds"),
            "result_path": str(path),
        }
    )

results_df = pd.DataFrame(rows)
display(results_df)

run_status_df = pd.DataFrame(json.loads(SUMMARY_PATH.read_text(encoding="utf-8")))
run_status_df

,model,benchmark,samples,success_count,failure_count,wall_clock_seconds,result_path
0,dataset-smoke-model,cityscapes,1,1,0,0.065927,C:\Users\Samuel\Documents\GitHub\transformers\...
1,dataset-smoke-model,conceptual_captions,1,1,0,0.742977,C:\Users\Samuel\Documents\GitHub\transformers\...
2,dataset-smoke-model,conceptual_captions_caption,1,1,0,0.361234,C:\Users\Samuel\Documents\GitHub\transformers\...
3,dataset-smoke-model,dfdc,1,1,0,0.070502,C:\Users\Samuel\Documents\GitHub\transformers\...
4,dataset-smoke-model,docvqa,1,1,0,0.057196,C:\Users\Samuel\Documents\GitHub\transformers\...
5,dataset-smoke-model,fairface,1,1,0,0.051824,C:\Users\Samuel\Documents\GitHub\transformers\...
6,dataset-smoke-model,fashion_mnist,1,1,0,0.055268,C:\Users\Samuel\Documents\GitHub\transformers\...
7,dataset-smoke-model,flickr30k,1,1,0,0.052513,C:\Users\Samuel\Documents\GitHub\transformers\...
8,dataset-smoke-model,flickr30k_entities,1,1,0,0.052831,C:\Users\Samuel\Documents\GitHub\transformers\...
9,llava-gemma-2b,cityscapes,1,1,0,70.797038,C:\Users\Samuel\Documents\GitHub\transformers\...


,model,benchmark,status,error,traceback,num_samples,max_new_tokens,results_path
0,llava-gemma-2b,blip3o_60k,error,ValueError: Could not find two curated attribu...,"Traceback (most recent call last):\n File ""C:...",NaN,NaN,NaN
1,llava-gemma-2b,conceptual_captions,ok,NaN,NaN,1.0,16.0,C:\Users\Samuel\Documents\GitHub\transformers\...
2,llava-gemma-2b,conceptual_captions_caption,ok,NaN,NaN,1.0,24.0,C:\Users\Samuel\Documents\GitHub\transformers\...
3,llava-gemma-2b,cityscapes,ok,NaN,NaN,1.0,16.0,C:\Users\Samuel\Documents\GitHub\transformers\...
4,llava-gemma-2b,dfdc,ok,NaN,NaN,1.0,16.0,C:\Users\Samuel\Documents\GitHub\transformers\...
5,llava-gemma-2b,diffusiondb,error,RuntimeError: Dataset scripts are no longer su...,"Traceback (most recent call last):\n File ""C:...",NaN,NaN,NaN
6,llava-gemma-2b,docvqa,ok,NaN,NaN,1.0,16.0,C:\Users\Samuel\Documents\GitHub\transformers\...
7,llava-gemma-2b,fairface,ok,NaN,NaN,1.0,16.0,C:\Users\Samuel\Documents\GitHub\transformers\...
8,llava-gemma-2b,fashion_mnist,ok,NaN,NaN,1.0,16.0,C:\Users\Samuel\Documents\GitHub\transformers\...
9,llava-gemma-2b,flickr30k,ok,NaN,NaN,1.0,24.0,C:\Users\Samuel\Documents\GitHub\transformers\...
